# Tutorial 6: Signing with FROST


**Audience:** Completed Tutorial 5

**Prerequisites:** DKG, Schnorr signatures

**Learning goals:**
- Execute a complete FROST signing ceremony
- Verify the result is a valid BIP340 signature
- See share verification catch a bad signer


## What this notebook covers

1. Quick DKG setup (self-contained)
2. Choosing signers and message
3. Round 1: Nonce generation
4. Round 2: Signing
5. Signature aggregation
6. Manual BIP340 verification
7. Share verification: catching a bad signer
8. Exercise: sign with a different signer subset


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash

# Quick DKG setup (from Tutorial 5)
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()

for p in participants:
    p.generate_shares()

for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants
        if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)

for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants
        if other.index != p.index
    )
    p.derive_public_key(other_commitments)

for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants
        if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key
print(f"DKG complete. Public key: ({public_key.x}, ...)")


## Choose Signers and Message

In a 2-of-3 scheme, any 2 participants can sign. Here we pick
participants 1 and 2.


In [ ]:
# 2-of-3: participants 1 and 2 will sign
signers = [participants[0], participants[1]]
signer_indexes = tuple(p.index for p in signers)
message = b"Hello from FROST!"
print(f"Signers: {signer_indexes}")
print(f"Message: {message}")


## Round 1: Nonce Generation

Each signer generates a nonce pair (d, e) and publishes the commitments
(D, E) = (d·G, e·G). The nonce pair is single-use and must be kept secret.


In [ ]:
for signer in signers:
    signer.generate_nonce_pair()
    D, E = signer.nonce_commitment_pair
    print(f"P{signer.index} nonce commitments: D=({D.x}, ...), E=({E.x}, ...)")


## Nonce Commitment List

Collect all commitments in a list indexed by participant number.
Non-signer slots get placeholder values (they won't be accessed).


In [ ]:
# Build nonce commitment list indexed by participant (1-indexed, so pad index 0)
all_nonce_pairs = [None] * n
for signer in signers:
    all_nonce_pairs[signer.index - 1] = signer.nonce_commitment_pair
# Fill non-signers with dummy commitments (they won't be used)
for i in range(n):
    if all_nonce_pairs[i] is None:
        all_nonce_pairs[i] = (Point(), Point())
nonce_commitment_pairs = tuple(all_nonce_pairs)
print(f"Collected {len(nonce_commitment_pairs)} nonce commitment pairs")


## Round 2: Signing

Each signer computes their signature share using the FROST equation:
z_i = d_i + e_i·ρ_i + λ_i·s_i·c


In [ ]:
signature_shares = []
for signer in signers:
    z_i = signer.sign(message, nonce_commitment_pairs, signer_indexes)
    signature_shares.append(z_i)
    print(f"P{signer.index} signature share: {z_i}")


## Aggregation

The Aggregator combines signature shares into a single BIP340 signature.
With group commitments provided, it verifies each share before aggregating.


In [ ]:
aggregator = Aggregator(
    public_key, message,
    nonce_commitment_pairs, signer_indexes,
    group_commitments=participants[0].group_commitments,
)
signature_hex = aggregator.signature(tuple(signature_shares))
print(f"BIP340 signature ({len(signature_hex) // 2} bytes): {signature_hex[:40]}...")


## Manual BIP340 Verification

Verify the signature using the raw Schnorr equation: s·G == R + e·P.
This is exactly what a Bitcoin node does.


In [ ]:
sig_bytes = bytes.fromhex(signature_hex)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")

R = Point.lift_x(R_x)
e_bytes = tagged_hash(
    "BIP0340/challenge",
    sig_bytes[:32] + public_key.to_bytes_xonly() + message,
)
e = int.from_bytes(e_bytes, "big") % Q

# Verify: s·G == R + e·P
lhs = s * G
rhs = R + (e * public_key)
print(f"BIP340 verification: s·G == R + e·P? {lhs == rhs}")


## Share Verification: Catching a Bad Signer

Corrupt a signature share and show that the Aggregator detects it.
Without share verification, a single bad share would silently produce
an invalid group signature.


In [ ]:
# Corrupt the first share
bad_shares = (signature_shares[0] + 1,) + tuple(signature_shares[1:])

try:
    bad_aggregator = Aggregator(
        public_key, message,
        nonce_commitment_pairs, signer_indexes,
        group_commitments=participants[0].group_commitments,
    )
    bad_aggregator.signature(bad_shares)
    print("ERROR: should have raised ValueError")
except ValueError as exc:
    print(f"Caught bad signer: {exc}")


## Exercise

Sign the same message with a different 2-of-3 subset (participants 2 and 3).
Verify that both signatures are valid but different.


In [ ]:
# Sign with participants 2 and 3
signers2 = [participants[1], participants[2]]
signer_indexes2 = tuple(p.index for p in signers2)

for signer in signers2:
    signer.generate_nonce_pair()

all_nonce_pairs2 = [None] * n
for signer in signers2:
    all_nonce_pairs2[signer.index - 1] = signer.nonce_commitment_pair
for i in range(n):
    if all_nonce_pairs2[i] is None:
        all_nonce_pairs2[i] = (Point(), Point())
nonce_commitment_pairs2 = tuple(all_nonce_pairs2)

sig_shares2 = []
for signer in signers2:
    z_i = signer.sign(message, nonce_commitment_pairs2, signer_indexes2)
    sig_shares2.append(z_i)

agg2 = Aggregator(public_key, message, nonce_commitment_pairs2, signer_indexes2)
sig2_hex = agg2.signature(tuple(sig_shares2))

print(f"Signature 1: {signature_hex[:32]}...")
print(f"Signature 2: {sig2_hex[:32]}...")
print(f"Different? {signature_hex != sig2_hex}")

# Verify signature 2
sig2_bytes = bytes.fromhex(sig2_hex)
R2_x = int.from_bytes(sig2_bytes[:32], "big")
s2 = int.from_bytes(sig2_bytes[32:], "big")
R2 = Point.lift_x(R2_x)
e2_bytes = tagged_hash("BIP0340/challenge", sig2_bytes[:32] + public_key.to_bytes_xonly() + message)
e2 = int.from_bytes(e2_bytes, "big") % Q
print(f"Signature 2 valid? {s2 * G == R2 + (e2 * public_key)}")


## Extension

FROST nonces are single-use. If a signer reuses a nonce pair across two
signing sessions, the attacker can extract the signer's aggregate share
(similar to the nonce reuse attack in Tutorial 3, but now targeting the
individual share rather than the full private key).
